# CS131 — GH Archive Exploration
Kernel: **CS131 (venv)**. This notebook peeks at ONE hour of GH Archive to
understand the schema. We do NOT load the full dataset here — that's what CLI
streaming (Phase 1) and PySpark (Phase 3) are for.

In [1]:
import gzip, json, pandas as pd
from pathlib import Path
from collections import Counter

# Resolve repo root robustly (works no matter the kernel's cwd)
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SAMPLE = ROOT / "data/sample/2024-01-15-15.json.gz"   # one hour, ~267k events
print("sample:", SAMPLE)

sample: /home/ishigaki-cs6/final_project/data/sample/2024-01-15-15.json.gz


## 1. Read the sample into a DataFrame (one hour fits in RAM)

In [2]:
df = pd.read_json(SAMPLE, lines=True, compression="gzip")
print(df.shape)
df.head(3)

(266871, 8)


,id,type,actor,repo,payload,public,created_at,org
0,34834865479,CreateEvent,"{'id': 140599107, 'login': 'prasham04', 'displ...","{'id': 743589745, 'name': 'prasham04/SImon-gam...","{'ref': None, 'ref_type': 'repository', 'maste...",True,2024-01-15 15:00:00+00:00,NaN
1,34834865487,CreateEvent,"{'id': 49699333, 'login': 'dependabot[bot]', '...","{'id': 223893722, 'name': 'dm-drogeriemarkt/li...",{'ref': 'dependabot/npm_and_yarn/react-router-...,True,2024-01-15 15:00:00+00:00,"{'id': 26007276, 'login': 'dm-drogeriemarkt', ..."
2,34834865488,PushEvent,"{'id': 118571769, 'login': 'abdelemjidessaid',...","{'id': 741908301, 'name': 'abdelemjidessaid/al...","{'repository_id': 741908301, 'push_id': 166455...",True,2024-01-15 15:00:00+00:00,NaN


## 2. Schema / columns

In [3]:
print(df.columns.tolist())
df.dtypes

['id', 'type', 'actor', 'repo', 'payload', 'public', 'created_at', 'org']


id                          int64
type                          str
actor                      object
repo                       object
payload                    object
public                       bool
created_at    datetime64[us, UTC]
org                        object
dtype: object

## 3. Event-type distribution

In [4]:
df["type"].value_counts()

type
PushEvent                        169030
CreateEvent                       28616
PullRequestEvent                  18272
IssueCommentEvent                 11687
WatchEvent                         9548
DeleteEvent                        7556
PullRequestReviewEvent             7217
PullRequestReviewCommentEvent      4311
IssuesEvent                        4170
ForkEvent                          2373
ReleaseEvent                       1247
PublicEvent                         891
CommitCommentEvent                  815
MemberEvent                         724
GollumEvent                         414
Name: count, dtype: int64

## 4. Flatten the nested repo/actor structs

In [5]:
df["repo_name"]  = df["repo"].apply(lambda r: r.get("name")  if isinstance(r, dict) else None)
df["actor_login"] = df["actor"].apply(lambda a: a.get("login") if isinstance(a, dict) else None)
df[["type","created_at","repo_name","actor_login"]].head()

,type,created_at,repo_name,actor_login
0,CreateEvent,2024-01-15 15:00:00+00:00,prasham04/SImon-game,prasham04
1,CreateEvent,2024-01-15 15:00:00+00:00,dm-drogeriemarkt/lisa,dependabot[bot]
2,PushEvent,2024-01-15 15:00:00+00:00,abdelemjidessaid/alx-backend-python,abdelemjidessaid
3,PushEvent,2024-01-15 15:00:00+00:00,oss333ulf/Projcts9,oss333ulf
4,PushEvent,2024-01-15 15:00:00+00:00,k-46/nand2Tetris-part-1,k-46


## 5. Prototype the AI/ML-repo classifier

In [6]:
import re
AI_PAT = re.compile(r"(llm|gpt|pytorch|tensorflow|langchain|diffus|transformer|"
                    r"huggingface|openai|deep.?learning|neural|machine.?learning|"
                    r"agentic|rag-)", re.I)
df["is_ai"] = df["repo_name"].fillna("").str.contains(AI_PAT)
print("AI/ML-repo events this hour:", int(df["is_ai"].sum()), "/", len(df))
df.loc[df["is_ai"], "repo_name"].value_counts().head(15)

AI/ML-repo events this hour: 2208 / 266871


/tmp/ipykernel_198801/654194796.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["is_ai"] = df["repo_name"].fillna("").str.contains(AI_PAT)


repo_name
rmayr/rediffusion                  124
pytorch/pytorch                    120
huggingface/transformers            53
huggingface/diffusers               52
huggingface/candle                  28
neuralmagic/deepsparse              21
langchain-ai/langchainjs            20
tsugumi-sys/SA-ConvLSTM-Pytorch     19
Lightning-AI/pytorch-lightning      19
jurisgpt/twg-criminal-law           18
lucidrains/equiformer-pytorch       16
tazgoulart/deep-learning            15
aaamoon/copilot-gpt4-service        13
juananmora/seleniumchatgpt4         13
567-labs/fastllm                    12
Name: count, dtype: int64

## Next steps
- Phase 1: run the timed CLI commands in `1_profile/profiling.txt` on many hours.
- Phase 2: prove pandas OOM with the `ulimit` recipe in `2_breaking/breaking.txt`.
- Phase 3: same logic at scale in `3_scaling/spark_job.py` on Dataproc.